# VayuSwarm — YOLOv8 Training on VisDrone Real Data

**Project:** VayuSwarm AI-Powered Swarm Drone Software  
**Dataset:** VisDrone real aerial drone images (540 train + 111 val + 549 test)  
**Classes:** person, car, truck, motorcycle, bicycle (5 classes)  
**Source:** [HuggingFace - kilanisainikhil/VayuSwarm](https://huggingface.co/kilanisainikhil/VayuSwarm)  
**Output:** Trained model auto-pushed to [GitHub - ved354/swam](https://github.com/ved354/swam)

### Setup
1. **Kaggle Settings → Accelerator → GPU T4 x2** (or P100)
2. **Kaggle Settings → Secrets → Add these 2 secrets:**
   - `HF_TOKEN` — HuggingFace token (to download dataset)
   - `GIT_TOKEN` — GitHub PAT (to push trained model)
3. Run all cells

## Cell 1: Install Dependencies

In [ ]:
!pip install -q ultralytics huggingface_hub

import os, json, shutil, subprocess, tempfile
from pathlib import Path
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Cell 2: Configuration

In [ ]:
# ── Load Secrets from Kaggle ──
from kaggle_secrets import UserSecretsClient

HF_TOKEN = ""
GIT_TOKEN = ""

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print("HF_TOKEN from env" if HF_TOKEN else "WARNING: No HF_TOKEN")

try:
    GIT_TOKEN = secrets.get_secret("GIT_TOKEN")
    print("GIT_TOKEN loaded from Kaggle Secrets")
except Exception:
    GIT_TOKEN = os.environ.get("GIT_TOKEN", "")
    print("GIT_TOKEN from env" if GIT_TOKEN else "WARNING: No GIT_TOKEN — model won't auto-push to GitHub")

# ── Hugging Face (dataset source) ──
HF_REPO = "kilanisainikhil/VayuSwarm"

# ── GitHub (model output destination) ──
GIT_REPO  = "https://github.com/ved354/swam.git"
GIT_USER  = "ved354"
GIT_EMAIL = "ved354@users.noreply.github.com"

# ── Training Config ──
MODEL_BASE = "yolov8n.pt"       # Pre-trained COCO backbone (nano)
EPOCHS     = 100                 # Training epochs
BATCH_SIZE = 32                  # Batch size (reduce to 16 if OOM)
IMG_SIZE   = 640                 # Input resolution
PATIENCE   = 20                  # Early stopping patience
DEVICE     = 0                   # GPU 0

# ── Paths ──
WORK_DIR    = Path("/kaggle/working")
DATASET_DIR = WORK_DIR / "datasets" / "VisDrone"
OUTPUT_DIR  = WORK_DIR / "runs"

# ── Classes ──
CLASSES = ["person", "car", "truck", "motorcycle", "bicycle"]

print(f"\nClasses ({len(CLASSES)}): {CLASSES}")
print(f"Training: {EPOCHS} epochs, batch {BATCH_SIZE}, {IMG_SIZE}px")
print(f"Output: {GIT_REPO}")

## Cell 3: Download Dataset from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

train_img_dir = DATASET_DIR / "images" / "train"

if train_img_dir.exists() and len(list(train_img_dir.glob("*.jpg"))) > 100:
    print(f"Dataset already exists at {DATASET_DIR}")
else:
    print(f"Downloading VisDrone dataset from {HF_REPO}...")
    local_path = snapshot_download(
        repo_id=HF_REPO,
        repo_type="model",
        allow_patterns=["datasets/VisDrone/**"],
        local_dir=str(WORK_DIR),
        token=HF_TOKEN if HF_TOKEN else None,
    )
    print(f"Downloaded to {local_path}")

# Verify
print(f"\nDataset contents:")
for split in ["train", "val", "test"]:
    imgs = len(list((DATASET_DIR / "images" / split).glob("*")))
    lbls = len(list((DATASET_DIR / "labels" / split).glob("*")))
    print(f"  {split:6s}: {imgs:4d} images, {lbls:5d} labels")

## Cell 4: Create data.yaml

In [ ]:
import yaml

data_config = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(CLASSES),
    "names": CLASSES,
}

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"data.yaml written to {yaml_path}")
print()
with open(yaml_path) as f:
    print(f.read())

## Cell 5: Train YOLOv8

In [ ]:
from ultralytics import YOLO

print(f"Starting YOLOv8 training...")
print(f"  Base model:  {MODEL_BASE}")
print(f"  Dataset:     VisDrone (real aerial images)")
print(f"  Epochs:      {EPOCHS}")
print(f"  Batch size:  {BATCH_SIZE}")
print(f"  Image size:  {IMG_SIZE}")
print()

model = YOLO(MODEL_BASE)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="patrol_rgb",
    exist_ok=True,
    # Augmentation tuned for aerial/drone patrol
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    erasing=0.4,
    # Optimizer
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    # Save
    save=True,
    plots=True,
    verbose=True,
)

SAVE_DIR = Path(results.save_dir)
print(f"\nTraining complete!")
print(f"Model saved to: {SAVE_DIR}")

## Cell 6: Evaluate on Test Set

In [ ]:
best_pt = SAVE_DIR / "weights" / "best.pt"
best_model = YOLO(str(best_pt))

metrics = best_model.val(data=str(yaml_path), split="test")

test_count = len(list((DATASET_DIR / "images" / "test").glob("*")))
print(f"\nTest Results ({test_count} real images):")
print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")

# Per-class breakdown
if hasattr(metrics.box, "ap_class_index"):
    print(f"\n  Per-class mAP50:")
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        cls_name = CLASSES[int(cls_idx)] if int(cls_idx) < len(CLASSES) else f"cls_{cls_idx}"
        print(f"    {cls_name:15s}  {metrics.box.ap50[i]:.4f}")

## Cell 7: Export & Save Metadata

In [ ]:
# Export ONNX
print("Exporting to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
print(f"ONNX exported: {onnx_path}")

# Save metadata
metadata = {
    "model_name": "vayuswarm_patrol_rgb",
    "base_model": MODEL_BASE,
    "dataset": "VisDrone (real aerial drone images)",
    "dataset_source": f"https://huggingface.co/{HF_REPO}",
    "num_classes": len(CLASSES),
    "classes": {str(i): name for i, name in enumerate(CLASSES)},
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "train_images": len(list((DATASET_DIR / "images" / "train").glob("*"))),
        "val_images": len(list((DATASET_DIR / "images" / "val").glob("*"))),
        "test_images": len(list((DATASET_DIR / "images" / "test").glob("*"))),
    },
    "metrics": {
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    },
}

metadata_path = SAVE_DIR / "class_mapping.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to: {metadata_path}")
print(json.dumps(metadata, indent=2))

## Cell 8: Auto-Push Trained Model to GitHub

Pushes `best.pt`, `last.pt`, `class_mapping.json`, ONNX export, and training plots  
to **https://github.com/ved354/swam** → `models/yolo/`

In [ ]:
if not GIT_TOKEN:
    print("No GIT_TOKEN found. Add it in Kaggle Settings -> Secrets.")
    print(f"Model saved locally at: {SAVE_DIR}")
else:
    try:
        # Build authenticated URL
        auth_url = GIT_REPO.replace("https://", f"https://{GIT_USER}:{GIT_TOKEN}@")
        clone_dir = Path(tempfile.mkdtemp()) / "swam"

        print(f"Pushing trained model to GitHub: {GIT_REPO}")
        print()

        # Clone repo (shallow)
        subprocess.check_call(["git", "clone", "--depth", "1", auth_url, str(clone_dir)])
        print("Cloned repo")

        # Create target directory
        model_dir = clone_dir / "models" / "yolo"
        model_dir.mkdir(parents=True, exist_ok=True)

        # Copy model files
        pushed_files = []
        for src, name in [
            (best_pt, "best.pt"),
            (SAVE_DIR / "weights" / "last.pt", "last.pt"),
            (metadata_path, "class_mapping.json"),
        ]:
            if src.exists():
                shutil.copy2(src, model_dir / name)
                size_mb = src.stat().st_size / 1024 / 1024
                pushed_files.append(name)
                print(f"  Copied {name} ({size_mb:.1f} MB)")

        # Copy ONNX
        for onnx_file in SAVE_DIR.glob("**/*.onnx"):
            shutil.copy2(onnx_file, model_dir / onnx_file.name)
            pushed_files.append(onnx_file.name)
            print(f"  Copied {onnx_file.name}")

        # Copy training plots
        plots_dest = model_dir / "plots"
        plots_dest.mkdir(exist_ok=True)
        for img in SAVE_DIR.glob("*.png"):
            shutil.copy2(img, plots_dest / img.name)
        print(f"  Copied training plots")

        # Git commit and push
        git_cmds = [
            ["git", "config", "user.name", GIT_USER],
            ["git", "config", "user.email", GIT_EMAIL],
            ["git", "add", "models/yolo/"],
            ["git", "commit", "-m",
             f"Add trained YOLOv8 patrol model — mAP50={metrics.box.map50:.4f}, "
             f"{len(CLASSES)} classes, {EPOCHS} epochs"],
            ["git", "push", "origin", "main"],
        ]

        for cmd in git_cmds:
            subprocess.check_call(cmd, cwd=str(clone_dir))

        print(f"\nModel pushed to: {GIT_REPO}")
        print(f"Files: {', '.join(pushed_files)}")
        print(f"Check: https://github.com/ved354/swam/tree/main/models/yolo")

    except Exception as e:
        print(f"GitHub push failed: {e}")
        print(f"Model saved locally at: {SAVE_DIR}")
        print(f"Download from Kaggle Output tab instead.")

## Cell 9: Summary

In [ ]:
print(f"{'='*60}")
print(f"VayuSwarm YOLOv8 Training Complete!")
print(f"{'='*60}")
print(f"Model:     {best_pt}")
print(f"Classes:   {CLASSES}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"")
print(f"GitHub:    https://github.com/ved354/swam/tree/main/models/yolo")
print(f"{'='*60}")